# 02 — Training Benchmark: Lance vs Parquet

**Purpose:** Quantify the data loading and training throughput gap between Lance (inline JPEG bytes, blob layout) and Delta/Parquet (image path references, separate Volumes files) at three dataset sizes.

**Prerequisite:** Run `01_create_benchmark_datasets.ipynb` for the target `size` tier first.

## Structure

| Section | What it measures |
|---------|------------------|
| **1 — Data Loading Throughput** | Raw samples/sec from each format before any GPU training |
| **2 — CNN Training Benchmark** | End-to-end ResNet-50 + FPN training throughput, GPU utilization, epoch wall time |
| **3 — Summary Table** | Side-by-side comparison; logged to MLflow |

**Key difference between formats during training:**
- **Lance** — `ray.data.read_lance` reads image bytes directly from the blob layout; zero additional I/O hop
- **Parquet** — `ray.data.read_databricks_tables` reads the Delta table (metadata only); a `map` step then reads each JPEG file from Volumes individually — 64 concurrent file reads per batch of 64

In [ ]:
# Install packages BEFORE any ray.init or @ray_launch call
%pip install -qU ray[all]==2.54.0 lance==0.17.0 pyarrow>=16.0 databricks-sdk>=0.49.0 "mlflow<3.0,>=2.17"
# Install serverless_gpu from your Workspace — update the path to match your environment
# %pip install '/Workspace/Users/<your-user>/ray-on-databricks-rct/distributed-training/XGBoost/databricks.serverless_gpu-0.5.3-py3-none-any.whl'
dbutils.library.restartPython()

In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size",         "10k",          ["10k", "100k", "1m"], "Dataset Size")
dbutils.widgets.text("catalog",          "main",         "UC Catalog")
dbutils.widgets.text("schema",           "ml_benchmark", "UC Schema")
dbutils.widgets.text("volume",           "bdd100k",      "UC Volume")
dbutils.widgets.text("warehouse_id",     "",             "SQL Warehouse ID")
dbutils.widgets.text("mlflow_experiment","",             "MLflow Experiment (leave blank for default)")

size         = dbutils.widgets.get("size")
catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")
volume       = dbutils.widgets.get("volume")
warehouse_id = dbutils.widgets.get("warehouse_id")

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
username      = notebook_path.split("/")[2]
mlflow_exp    = dbutils.widgets.get("mlflow_experiment") or f"/Users/{username}/lance-vs-parquet-benchmark"

base_vol       = f"/Volumes/{catalog}/{schema}/{volume}"
lance_path     = f"{base_vol}/bdd100k_lance_{size}"
parquet_img_dir = f"{base_vol}/bdd100k_parquet_{size}"
delta_table    = f"{catalog}.{schema}.bdd100k_parquet_{size}"
ray_tmp_path   = f"/Volumes/{catalog}/{schema}/ray_tmp"

print(f"Size tier     : {size}")
print(f"Lance path    : {lance_path}")
print(f"Delta table   : {delta_table}")
print(f"MLflow exp    : {mlflow_exp}")

In [ ]:
import os

# Credentials — set before @ray_launch so workers inherit them
os.environ['DATABRICKS_HOST']  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ['DATABRICKS_TOKEN'] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Capture driver-side values for closures (small scalars only)
_db_host      = os.environ['DATABRICKS_HOST']
_db_token     = os.environ['DATABRICKS_TOKEN']
_warehouse_id = warehouse_id
_catalog      = catalog
_schema       = schema
_delta_table  = delta_table
_lance_path   = lance_path
_ray_tmp      = ray_tmp_path
_mlflow_exp   = mlflow_exp
_size         = size

In [ ]:
# Provision or reuse a SQL Warehouse for read_databricks_tables
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

WAREHOUSE_NAME  = "ray-benchmark-warehouse"
AUTO_STOP_MINS  = 10
CLUSTER_SIZE    = "Small"

if not _warehouse_id:
    w = WorkspaceClient()
    _warehouse_id = None
    for wh in w.warehouses.list():
        if wh.name == WAREHOUSE_NAME:
            _warehouse_id = wh.id
            if wh.state in (State.STOPPED, State.STOPPING):
                w.warehouses.start(wh.id).result()
            elif wh.state == State.STARTING:
                w.warehouses.get_and_wait(wh.id)
            print(f"Using warehouse '{WAREHOUSE_NAME}' ({_warehouse_id})")
            break

    if not _warehouse_id:
        created = w.warehouses.create(
            name=WAREHOUSE_NAME,
            cluster_size=CLUSTER_SIZE,
            auto_stop_mins=AUTO_STOP_MINS,
            enable_serverless_compute=True,
            min_num_clusters=1,
            max_num_clusters=1,
        ).result()
        _warehouse_id = created.id
        print(f"Created warehouse '{WAREHOUSE_NAME}' ({_warehouse_id})")
else:
    print(f"Using warehouse from widget: {_warehouse_id}")

---

## Section 1 — Data Loading Throughput

Measures raw data loading speed for each format — no model training, no GPU compute.

**Parquet pipeline:**
1. `ray.data.read_databricks_tables` → reads Delta table metadata rows via SQL Warehouse
2. `.map(read_image_from_path)` → reads each JPEG file from Volumes (simulates the DataLoader I/O hop)

**Lance pipeline:**
1. `ray.data.read_lance` → reads image bytes directly from blob layout (no additional I/O hop)

Both pipelines produce batches of `{"image": <bytes>, "bbox_labels": <str>}` and are timed over a full pass.

In [ ]:
import ray


@ray.remote(num_cpus=1)
class ThroughputBenchmark:
    """Measures data loading throughput for both storage formats."""

    def run(self):
        import os, time, ray

        os.environ['DATABRICKS_HOST']  = _db_host
        os.environ['DATABRICKS_TOKEN'] = _db_token

        results = {}

        # ── Parquet ──────────────────────────────────────────────────────────
        try:
            ds_parquet = ray.data.read_databricks_tables(
                warehouse_id=_warehouse_id,
                catalog=_catalog,
                schema=_schema,
                query=f"SELECT frame_id, image_path, bbox_labels FROM {_delta_table}",
            )
        except Exception:
            # Fallback: read the Delta Parquet files directly
            ds_parquet = ray.data.read_delta_sharing_tables  # not available — use read_parquet
            ds_parquet = None

        if ds_parquet is not None:
            # Map: read JPEG bytes from Volumes path (simulates DataLoader I/O hop)
            def read_image_from_path(row):
                with open(row["image_path"], "rb") as f:
                    return {"image": f.read(), "bbox_labels": row["bbox_labels"]}

            ds_parquet = ds_parquet.map(read_image_from_path)

            t0 = time.time()
            n_parquet = 0
            for batch in ds_parquet.iter_batches(batch_size=64, batch_format="numpy"):
                n_parquet += len(batch["image"])
            elapsed = time.time() - t0
            results["parquet"] = {
                "samples_per_sec": round(n_parquet / elapsed, 1),
                "total_samples":   n_parquet,
                "elapsed_sec":     round(elapsed, 2),
            }
        else:
            results["parquet"] = {"error": "read_databricks_tables failed — set warehouse_id widget"}

        # ── Lance ─────────────────────────────────────────────────────────────
        ds_lance = ray.data.read_lance(
            _lance_path,
            columns=["image", "bbox_labels"],
        )

        t0 = time.time()
        n_lance = 0
        for batch in ds_lance.iter_batches(batch_size=64, batch_format="numpy"):
            n_lance += len(batch["image"])
        elapsed = time.time() - t0
        results["lance"] = {
            "samples_per_sec": round(n_lance / elapsed, 1),
            "total_samples":   n_lance,
            "elapsed_sec":     round(elapsed, 2),
        }

        return results

In [ ]:
from serverless_gpu.ray import ray_launch


@ray_launch(gpus=4, gpu_type='A10', remote=True)
def run_throughput_benchmark():
    runner = ThroughputBenchmark.remote()
    return ray.get(runner.run.remote())


throughput_results = run_throughput_benchmark.distributed()

print("\n── Data Loading Throughput ──────────────────────────")
for fmt, r in throughput_results.items():
    if "error" in r:
        print(f"  {fmt:8s}: {r['error']}")
    else:
        print(f"  {fmt:8s}: {r['samples_per_sec']:>10,.1f} samples/sec  "
              f"({r['total_samples']:,} samples in {r['elapsed_sec']}s)")

---

## Section 2 — CNN Training Benchmark

Trains a ResNet-50 + FPN object detection model (10-category BDD100K labels) on each storage format using `TorchTrainer` with DDP across 4 GPU workers.

**3 epochs** — enough to reach steady-state throughput without waiting for convergence.

Each format run is logged as a separate MLflow run under a shared parent experiment, with:
- `format` and `dataset_size` as parameters
- `samples_per_sec`, `data_load_ms_per_batch`, `gpu_util_pct`, `epoch_wall_time_sec` as metrics per epoch

In [ ]:
# Model definition — shared across both format runs
# Uses torchvision's Faster R-CNN with ResNet-50 + FPN backbone
# (matches the architecture described in 02_cnn_training.ipynb)

def build_detection_model(num_classes=11):  # 10 BDD100K categories + background
    import torchvision
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def decode_batch_to_tensors(batch, device):
    """Decode JPEG bytes to normalized image tensors + parse bbox labels."""
    import io, json
    import torch
    import torchvision.transforms.functional as TF
    from PIL import Image

    images, targets = [], []
    for jpeg_bytes, bbox_json in zip(batch["image"], batch["bbox_labels"]):
        img = Image.open(io.BytesIO(bytes(jpeg_bytes))).convert("RGB")
        img_tensor = TF.to_tensor(img).to(device)  # CxHxW, float32 in [0,1]
        images.append(img_tensor)

        bboxes = json.loads(bbox_json) if isinstance(bbox_json, str) else (bbox_json or [])
        if bboxes:
            boxes  = torch.tensor([[b["x1"], b["y1"], b["x2"], b["y2"]] for b in bboxes],
                                   dtype=torch.float32, device=device)
            labels = torch.ones(len(bboxes), dtype=torch.int64, device=device)  # simplified: all class 1
        else:
            boxes  = torch.zeros((0, 4), dtype=torch.float32, device=device)
            labels = torch.zeros(0,       dtype=torch.int64,   device=device)
        targets.append({"boxes": boxes, "labels": labels})

    return images, targets

In [ ]:
def train_fn_per_worker(config):
    """Per-worker DDP training function — executed by TorchTrainer on each GPU worker."""
    import os, time
    import torch
    import ray.train
    import ray.train.torch
    from mlflow.tracking import MlflowClient

    torch.set_num_threads(1)  # prevent intra-op thread contention across workers

    device    = ray.train.torch.get_device()
    rank      = ray.train.get_context().get_world_rank()
    run_id    = config["mlflow_run_id"]
    n_epochs  = config["num_epochs"]
    lr        = config["lr"]

    model     = build_detection_model(num_classes=11).to(device)
    model     = ray.train.torch.prepare_model(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

    shard = ray.train.get_dataset_shard("train")
    client = MlflowClient() if rank == 0 else None

    for epoch in range(n_epochs):
        model.train()
        t0 = time.time()
        n_samples       = 0
        total_load_ms   = 0.0

        for batch in shard.iter_batches(batch_size=config["batch_size"], batch_format="numpy"):
            t_load = time.time()
            images, targets = decode_batch_to_tensors(batch, device)
            total_load_ms += (time.time() - t_load) * 1000

            loss_dict = model(images, targets)
            loss      = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            n_samples += len(images)

        epoch_time      = time.time() - t0
        samples_per_sec = n_samples / epoch_time if epoch_time > 0 else 0
        load_ms_per_batch = total_load_ms / max(1, n_samples // config["batch_size"])

        if client is not None:
            client.log_metric(run_id, "samples_per_sec",     samples_per_sec,  step=epoch)
            client.log_metric(run_id, "data_load_ms_per_batch", load_ms_per_batch, step=epoch)
            client.log_metric(run_id, "epoch_wall_time_sec", epoch_time,        step=epoch)

        ray.train.report({
            "samples_per_sec":       samples_per_sec,
            "data_load_ms_per_batch": load_ms_per_batch,
            "epoch_wall_time_sec":   epoch_time,
        })

In [ ]:
import mlflow

training_results = {}

for fmt in ["parquet", "lance"]:
    _fmt = fmt  # capture for closure

    @ray.remote(num_cpus=1)
    class TrainRunner:
        def run(self):
            import os, mlflow
            import ray
            import ray.data
            from ray.train.torch import TorchTrainer
            from ray.train import ScalingConfig, RunConfig

            os.environ['DATABRICKS_HOST']  = _db_host
            os.environ['DATABRICKS_TOKEN'] = _db_token

            mlflow.set_experiment(_mlflow_exp)

            # ── Load dataset for this format ──────────────────────────────
            if _fmt == "parquet":
                try:
                    ds = ray.data.read_databricks_tables(
                        warehouse_id=_warehouse_id,
                        catalog=_catalog,
                        schema=_schema,
                        query=f"SELECT frame_id, image_path, bbox_labels FROM {_delta_table}",
                    )
                    # Materialise image bytes from path references
                    def _read_image(row):
                        with open(row["image_path"], "rb") as f:
                            return {"image": f.read(), "bbox_labels": row.get("bbox_labels", "[]")}
                    ds = ds.map(_read_image)
                except Exception as e:
                    return {"error": str(e)}
            else:  # lance
                ds = ray.data.read_lance(
                    _lance_path,
                    columns=["image", "bbox_labels"],
                )

            # ── MLflow run ────────────────────────────────────────────────
            with mlflow.start_run(run_name=f"cnn_{_fmt}_{_size}") as run:
                mlflow.log_params({
                    "format":       _fmt,
                    "dataset_size": _size,
                    "model":        "fasterrcnn_resnet50_fpn",
                    "num_epochs":   3,
                    "batch_size":   64,
                    "num_workers":  4,
                })

                trainer = TorchTrainer(
                    train_loop_per_worker=train_fn_per_worker,
                    train_loop_config={
                        "lr":           1e-3,
                        "num_epochs":   3,
                        "batch_size":   64,
                        "mlflow_run_id": run.info.run_id,
                    },
                    scaling_config=ScalingConfig(
                        num_workers=4,
                        use_gpu=True,
                        resources_per_worker={"CPU": 4, "GPU": 1},
                    ),
                    datasets={"train": ds},
                    run_config=RunConfig(storage_path=_ray_tmp),
                )

                result = trainer.fit()

                mlflow.log_metrics({
                    "final_samples_per_sec":        result.metrics.get("samples_per_sec", 0),
                    "final_data_load_ms_per_batch": result.metrics.get("data_load_ms_per_batch", 0),
                    "final_epoch_wall_time_sec":    result.metrics.get("epoch_wall_time_sec", 0),
                })

                return {
                    "run_id":               run.info.run_id,
                    "samples_per_sec":      result.metrics.get("samples_per_sec", 0),
                    "data_load_ms_per_batch": result.metrics.get("data_load_ms_per_batch", 0),
                    "epoch_wall_time_sec":  result.metrics.get("epoch_wall_time_sec", 0),
                }

    @ray_launch(gpus=4, gpu_type='A10', remote=True)
    def run_training():
        runner = TrainRunner.remote()
        return ray.get(runner.run.remote())

    print(f"\nTraining on {fmt} dataset ({size}) ...")
    training_results[fmt] = run_training.distributed()
    print(f"  {fmt}: {training_results[fmt]}")

---

## Section 3 — Summary Table

Side-by-side comparison of both formats at the selected size tier.

**Expected pattern:**
- Lance `data_load_ms_per_batch` should be significantly lower — no per-image Volumes I/O hop
- Lance `samples_per_sec` should be higher — GPU stays busy instead of waiting for data
- The gap widens at `100k` and `1m` where Volumes read latency compounds across more batches

In [ ]:
import pandas as pd

rows = []
for fmt in ["parquet", "lance"]:
    tr = training_results.get(fmt, {})
    thr = throughput_results.get(fmt, {})
    rows.append({
        "Format":                   fmt,
        "Dataset Size":             size,
        "Data Load (samples/sec)": thr.get("samples_per_sec", "—"),
        "Train (samples/sec)":     tr.get("samples_per_sec", "—"),
        "Data Load (ms/batch)":    tr.get("data_load_ms_per_batch", "—"),
        "Epoch Wall Time (s)":     tr.get("epoch_wall_time_sec", "—"),
        "MLflow Run ID":           tr.get("run_id", "—"),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

# Compute speedup
lance_row   = summary_df[summary_df["Format"] == "lance"]
parquet_row = summary_df[summary_df["Format"] == "parquet"]

if not lance_row.empty and not parquet_row.empty:
    lance_tps   = lance_row["Train (samples/sec)"].values[0]
    parquet_tps = parquet_row["Train (samples/sec)"].values[0]
    if isinstance(lance_tps, (int, float)) and isinstance(parquet_tps, (int, float)) and parquet_tps > 0:
        print(f"\nLance is {lance_tps / parquet_tps:.2f}x faster than Parquet "
              f"at size={size} (training throughput)")